# Coffee House: LoRA FP16 en Colab

Experimento reproducible sobre Llama 3.2 1B Instruct, 560/140 conversaciones sintéticas. Consultar `docs/Experimento-LoRA.md` para resultados y limitaciones. Seleccionar GPU; guardar HF_TOKEN en Secretos. Este notebook no publica credenciales ni pesos. El entrenamiento queda deshabilitado hasta cambiar RUN_TRAINING.


In [ ]:
import torch
assert torch.cuda.is_available(), "Selecciona una GPU de Colab"
print(torch.cuda.get_device_name(0))
RUN_TRAINING = False


In [ ]:
%pip install transformers==5.18.0 peft==0.21.2 accelerate==1.15.0 datasets==5.0.1


Si un paquete preinstalado opcional causa un error de compatibilidad, usar un entorno nuevo y conservar PyTorch CUDA de Colab. Las versiones se fijan para reproducir el protocolo original.


In [ ]:
from pathlib import Path
import hashlib, json, math
from google.colab import files, userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model
from datasets import Dataset
set_seed(42)
MODEL = "meta-llama/Llama-3.2-1B-Instruct"
REVISION = "9213176726f574b556790deb65791e0c5aa438b6"
OUTPUT = Path("/content/coffee-house-lora-reproduction")
expected = {"train.jsonl":"c771cbaf7bee653125e19879be3f4ab92c6f78c9717cccce6ba6f947742b14b4", "validation.jsonl":"0978b0b76c41a2c4b64fb946eb5b9a264ffae3818736ae50cd46f18b8a49ed98"}
uploaded = files.upload()
for name, digest in expected.items():
    assert hashlib.sha256(uploaded[name]).hexdigest() == digest, "El corpus no coincide"
train = [json.loads(line) for line in uploaded["train.jsonl"].decode().splitlines()]
validation = [json.loads(line) for line in uploaded["validation.jsonl"].decode().splitlines()]
assert (len(train),len(validation)) == (560,140)
assert not ({r["family"] for r in train} & {r["family"] for r in validation})
assert all(r["synthetic"] is True for r in train+validation)


In [ ]:
from collections.abc import Mapping
class InvalidDataset(ValueError):
    pass

def encode_conversation(tokenizer, messages, *, max_length=1024, assistant_turns="all"):
    if assistant_turns not in {"all", "last"}:
        raise InvalidDataset("Selección de respuestas desconocida")
    def render(turns, generation=False):
        result = tokenizer.apply_chat_template(turns, tokenize=True,
            add_generation_prompt=generation, date_string="04 Oct 2026")
        return result["input_ids"] if isinstance(result, Mapping) else result
    full = render(messages)
    if len(full) > max_length:
        raise InvalidDataset("Ejemplo demasiado largo: excluir o reescribir, nunca truncar")
    labels = [-100] * len(full)
    for i, message in enumerate(messages):
        if message["role"] != "assistant":
            continue
        if assistant_turns == "last" and i != len(messages)-1:
            continue
        prefix = render(messages[:i], generation=True)
        through = render(messages[:i+1])
        if full[:len(prefix)] != prefix or full[:len(through)] != through or len(through) <= len(prefix):
            raise InvalidDataset("Plantilla no conserva los límites de la respuesta")
        labels[len(prefix):len(through)] = full[len(prefix):len(through)]
    if all(label == -100 for label in labels):
        raise InvalidDataset("Sin tokens supervisados")
    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}


class ResponseCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, examples):
        import torch

        length = max(len(e["input_ids"]) for e in examples)
        result = {}
        for field, padding in (("input_ids", self.pad_token_id), ("attention_mask", 0), ("labels", -100)):
            result[field] = torch.tensor([e[field]+[padding]*(length-len(e[field])) for e in examples])
        return result


In [ ]:
hf_token = userdata.get("HF_TOKEN")
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
encoded_train = [encode_conversation(tokenizer,r["messages"],assistant_turns="last") for r in train]
encoded_validation = [encode_conversation(tokenizer,r["messages"],assistant_turns="last") for r in validation]
print({"train":len(train),"validation":len(validation),"max_tokens":max(len(r["input_ids"]) for r in encoded_train+encoded_validation)})


In [ ]:
if RUN_TRAINING:
    OUTPUT.mkdir(exist_ok=False)
    torch.cuda.reset_peak_memory_stats()
    base = AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION,token=hf_token,dtype=torch.float16)
    base.config.use_cache = False
    config = LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=["q_proj","v_proj"],bias="none",task_type="CAUSAL_LM")
    model = get_peft_model(base,config).to("cuda")
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    assert all("lora_" in n for n,p in model.named_parameters() if p.requires_grad)
del hf_token


In [ ]:
if RUN_TRAINING:
    collator = ResponseCollator(tokenizer.pad_token_id)
    set_seed(42)
    args = TrainingArguments(output_dir=str(OUTPUT/"checkpoints"), num_train_epochs=2,
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
        learning_rate=0.0001, fp16=True, gradient_checkpointing=True, optim="adamw_torch",
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False, save_total_limit=2,
        report_to="none", logging_steps=10, seed=42, data_seed=42)
    trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_list(encoded_train),
        eval_dataset=Dataset.from_list(encoded_validation), data_collator=collator)
    result = trainer.train()
    model.save_pretrained(OUTPUT/"adapter")
    tokenizer.save_pretrained(OUTPUT/"tokenizer")
    assert math.isfinite(result.training_loss)
    record = {"model":MODEL,"revision":REVISION,"dataset_sha256":expected,"metrics":result.metrics,"history":trainer.state.log_history,"peak_allocated_bytes":torch.cuda.max_memory_allocated(),"peak_reserved_bytes":torch.cuda.max_memory_reserved()}
    (OUTPUT/"results.json").write_text(json.dumps(record,indent=2),encoding="utf8")
    print("Entrenamiento terminado. Evaluar calidad antes de usar el adaptador.")
else:
    print("Preparación terminada; entrenamiento deshabilitado.")


## Evaluación posterior

Comparar modelo base y adaptador con exactamente las mismas preguntas, hechos, parámetros y criterios. Revisar disponibilidad, precio total, información faltante y tono. Una pérdida menor no demuestra mejor experiencia. Conservar respuestas incorrectas y resultados completos; no atribuir al modelo las comprobaciones deterministas del backend.
